In [2]:
%run ./lehmer_pattern_avoid_v2.ipynb

In [3]:
import sage.combinat.permutation as permutation

# Column and Row statistic

In [4]:
def dyck_to_area_cells(dw):
    
    s = str(dw)
    n = len(s) // 2

    cells = []
    x, y = 0, 0

    for ch in s:
        if ch == '(': #up
            y += 1
        elif ch == ')': #right, column index+1, now height is y
            col = x + 1
            for row in range(col + 1, y + 1): #count col+1, ..., y for current column index 'col'.
                cells.append((col, row))
            x += 1
        else:
            raise ValueError("Wrong Dyck path")

    return cells


def column_stat(dw):
    
    s = str(dw)
    n = len(s) // 2
    cells = dyck_to_area_cells(dw)

    col_counts = [0] * n
    for col, row in cells:
        col_counts[col - 1] += 1

    return col_counts


def row_stat(dw):

    s = str(dw)
    n = len(s) // 2
    cells = dyck_to_area_cells(dw)

    row_counts = [0] * n
    for col, row in cells:
        row_counts[row - 1] += 1

    return row_counts

In [5]:
dyck_to_area_cells("((()))")

[(1, 2), (1, 3), (2, 3)]

In [6]:
print(column_stat("((()))"))
print(row_stat("((()))"))

[2, 1, 0]
[0, 1, 2]


In [7]:
# The column_stat is Lehmer of 312 avoiding, trivial as we proved bijection of 312 lehmer to it's own column_stat,
# Hence any other pattern --> its column_stat is actually a bijection between this pattern and 312 lehmer.
for dw in DyckWords(4):
    print(dw, dyck_to_area_cells(dw), column_stat(dw), row_stat(dw))

()()()() [] [0, 0, 0, 0] [0, 0, 0, 0]
()()(()) [(3, 4)] [0, 0, 1, 0] [0, 0, 0, 1]
()(())() [(2, 3)] [0, 1, 0, 0] [0, 0, 1, 0]
()(()()) [(2, 3), (3, 4)] [0, 1, 1, 0] [0, 0, 1, 1]
()((())) [(2, 3), (2, 4), (3, 4)] [0, 2, 1, 0] [0, 0, 1, 2]
(())()() [(1, 2)] [1, 0, 0, 0] [0, 1, 0, 0]
(())(()) [(1, 2), (3, 4)] [1, 0, 1, 0] [0, 1, 0, 1]
(()())() [(1, 2), (2, 3)] [1, 1, 0, 0] [0, 1, 1, 0]
(()()()) [(1, 2), (2, 3), (3, 4)] [1, 1, 1, 0] [0, 1, 1, 1]
(()(())) [(1, 2), (2, 3), (2, 4), (3, 4)] [1, 2, 1, 0] [0, 1, 1, 2]
((()))() [(1, 2), (1, 3), (2, 3)] [2, 1, 0, 0] [0, 1, 2, 0]
((())()) [(1, 2), (1, 3), (2, 3), (3, 4)] [2, 1, 1, 0] [0, 1, 2, 1]
((()())) [(1, 2), (1, 3), (2, 3), (2, 4), (3, 4)] [2, 2, 1, 0] [0, 1, 2, 2]
(((()))) [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)] [3, 2, 1, 0] [0, 1, 2, 3]


# Strip statistics (Two ways)

In [8]:
def strip_decomposition_from_topright(dw):
    
    remaining = set(dyck_to_area_cells(dw))
    strips = []

    while remaining:
        start = max(remaining, key=lambda cell: (cell[0] + cell[1]))

        strip = []
        c, r = start
        while (c, r) in remaining:
            strip.append((c, r))
            remaining.remove((c, r))
            c -= 1
            r -= 1

        strips.append(strip)

    return strips


def strip_decomposition_from_bottomleft(dw):

    remaining = set(dyck_to_area_cells(dw))
    strips = []

    while remaining:
        start = min(remaining, key=lambda cell: (cell[0] + cell[1]))

        strip = []
        c, r = start
        while (c, r) in remaining:
            strip.append((c, r))
            remaining.remove((c, r))
            c += 1
            r += 1

        strips.append(strip)

    return strips

def strip_stat_from_topright(dw):
    return [len(strip) for strip in strip_decomposition_from_topright(dw)]

def strip_stat_from_bottomleft(dw):
    return [len(strip) for strip in strip_decomposition_from_bottomleft(dw)] 

In [9]:
strip_stat_from_topright("((()))(())")

[1, 2, 1]

# Test on all patterns #

In [10]:
#rev of row stat all avoid 312 pattern, hmm, actually we somehow know this...
dw = DyckWords(4)
# dw = [ dw[2]]
for d in dw:
    print(row_stat(d),is_312_avoiding_code(row_stat(d)[::-1]))

[0, 0, 0, 0] True
[0, 0, 0, 1] True
[0, 0, 1, 0] True
[0, 0, 1, 1] True
[0, 0, 1, 2] True
[0, 1, 0, 0] True
[0, 1, 0, 1] True
[0, 1, 1, 0] True
[0, 1, 1, 1] True
[0, 1, 1, 2] True
[0, 1, 2, 0] True
[0, 1, 2, 1] True
[0, 1, 2, 2] True
[0, 1, 2, 3] True


In [12]:
dw = DyckWords(4)
# dw = [ dw[2]]
lehmer_set = set()
for d in dw:
    lehmer = dw_alpha_last(d)
    # lehmer132 = dw_beta_last(d)
    # lehmer213 = dw_gamma_last(d)
    # lehmer231 = dw_delta_last(d)
    # lehmer312 = dw_epsilon_last(d)
    # lehmer321 = dw_zeta_last(d)
    lehmer_set.add(tuple(lehmer))
    perm = permutation.from_lehmer_code(lehmer)
    colstat = column_stat(d)
    rowstat = row_stat(d)
    stripright = strip_stat_from_topright(d)
    stripleft = strip_stat_from_bottomleft(d)
    print(d, lehmer, perm, colstat,'row:' , rowstat, rowstat[::-1], sum(lehmer), len(dyck_to_area_cells(d)))
print(len(dw), len(lehmer_set))

()()()() [3, 2, 1, 0] [4, 3, 2, 1] [0, 0, 0, 0] row: [0, 0, 0, 0] [0, 0, 0, 0] 6 0
()()(()) [3, 2, 0, 0] [4, 3, 1, 2] [0, 0, 1, 0] row: [0, 0, 0, 1] [1, 0, 0, 0] 5 1
()(())() [3, 1, 1, 0] [4, 2, 3, 1] [0, 1, 0, 0] row: [0, 0, 1, 0] [0, 1, 0, 0] 5 1
()(()()) [3, 1, 0, 0] [4, 2, 1, 3] [0, 1, 1, 0] row: [0, 0, 1, 1] [1, 1, 0, 0] 4 2
()((())) [3, 0, 1, 0] [4, 1, 3, 2] [0, 2, 1, 0] row: [0, 0, 1, 2] [2, 1, 0, 0] 4 3
(())()() [2, 2, 1, 0] [3, 4, 2, 1] [1, 0, 0, 0] row: [0, 1, 0, 0] [0, 0, 1, 0] 5 1
(())(()) [2, 2, 0, 0] [3, 4, 1, 2] [1, 0, 1, 0] row: [0, 1, 0, 1] [1, 0, 1, 0] 4 2
(()())() [2, 1, 1, 0] [3, 2, 4, 1] [1, 1, 0, 0] row: [0, 1, 1, 0] [0, 1, 1, 0] 4 2
(()()()) [2, 0, 1, 0] [3, 1, 4, 2] [1, 1, 1, 0] row: [0, 1, 1, 1] [1, 1, 1, 0] 3 3
(()(())) [2, 1, 0, 0] [3, 2, 1, 4] [1, 2, 1, 0] row: [0, 1, 1, 2] [2, 1, 1, 0] 3 4
((()))() [1, 2, 1, 0] [2, 4, 3, 1] [2, 1, 0, 0] row: [0, 1, 2, 0] [0, 2, 1, 0] 4 3
((())()) [1, 0, 1, 0] [2, 1, 4, 3] [2, 1, 1, 0] row: [0, 1, 2, 1] [1, 2, 1, 0] 2 4
((()

In [13]:
dw = DyckWords(4)
lehmer_set = set()
for d in dw:
    # lehmer123 = dw_alpha_last(d)
    lehmer = dw_beta_last(d)
    # lehmer213 = dw_gamma_last(d)
    # lehmer231 = dw_delta_last(d)
    # lehmer312 = dw_epsilon_last(d)
    # lehmer321 = dw_zeta_last(d)
    lehmer_set.add(tuple(lehmer))
    perm = permutation.from_lehmer_code(lehmer)
    colstat = column_stat(d)
    rowstat = row_stat(d)
    stripright = strip_stat_from_topright(d)
    stripleft = strip_stat_from_bottomleft(d)
    print(d, lehmer, perm, colstat, rowstat, sum(lehmer), len(dyck_to_area_cells(d)))
print(len(dw), len(lehmer_set))

()()()() [0, 0, 0, 0] [1, 2, 3, 4] [0, 0, 0, 0] [0, 0, 0, 0] 0 0
()()(()) [1, 1, 1, 0] [2, 3, 4, 1] [0, 0, 1, 0] [0, 0, 0, 1] 3 1
()(())() [1, 1, 0, 0] [2, 3, 1, 4] [0, 1, 0, 0] [0, 0, 1, 0] 2 1
()(()()) [2, 2, 0, 0] [3, 4, 1, 2] [0, 1, 1, 0] [0, 0, 1, 1] 4 2
()((())) [2, 2, 1, 0] [3, 4, 2, 1] [0, 2, 1, 0] [0, 0, 1, 2] 5 3
(())()() [1, 0, 0, 0] [2, 1, 3, 4] [1, 0, 0, 0] [0, 1, 0, 0] 1 1
(())(()) [2, 1, 1, 0] [3, 2, 4, 1] [1, 0, 1, 0] [0, 1, 0, 1] 4 2
(()())() [2, 0, 0, 0] [3, 1, 2, 4] [1, 1, 0, 0] [0, 1, 1, 0] 2 2
(()()()) [3, 0, 0, 0] [4, 1, 2, 3] [1, 1, 1, 0] [0, 1, 1, 1] 3 3
(()(())) [3, 1, 1, 0] [4, 2, 3, 1] [1, 2, 1, 0] [0, 1, 1, 2] 5 4
((()))() [2, 1, 0, 0] [3, 2, 1, 4] [2, 1, 0, 0] [0, 1, 2, 0] 3 3
((())()) [3, 1, 0, 0] [4, 2, 1, 3] [2, 1, 1, 0] [0, 1, 2, 1] 4 4
((()())) [3, 2, 0, 0] [4, 3, 1, 2] [2, 2, 1, 0] [0, 1, 2, 2] 5 5
(((()))) [3, 2, 1, 0] [4, 3, 2, 1] [3, 2, 1, 0] [0, 1, 2, 3] 6 6
14 14


In [14]:
dw = DyckWords(4)
lehmer_set = set()
for d in dw:
    # lehmer123 = dw_alpha_last(d)
    # lehmer132 = dw_beta_last(d)
    lehmer = dw_gamma_last(d)
    # lehmer231 = dw_delta_last(d)
    # lehmer312 = dw_epsilon_last(d)
    # lehmer321 = dw_zeta_last(d)
    lehmer_set.add(tuple(lehmer))
    perm = permutation.from_lehmer_code(lehmer)
    colstat = column_stat(d)
    rowstat = row_stat(d)
    stripright = strip_stat_from_topright(d)
    stripleft = strip_stat_from_bottomleft(d)
    print(d, lehmer, perm, colstat, rowstat, sum(lehmer), len(dyck_to_area_cells(d)))
print(len(dw), len(lehmer_set))

()()()() [3, 2, 1, 0] [4, 3, 2, 1] [0, 0, 0, 0] [0, 0, 0, 0] 6 0
()()(()) [3, 2, 0, 0] [4, 3, 1, 2] [0, 0, 1, 0] [0, 0, 0, 1] 5 1
()(())() [3, 1, 1, 0] [4, 2, 3, 1] [0, 1, 0, 0] [0, 0, 1, 0] 5 1
()(()()) [3, 0, 1, 0] [4, 1, 3, 2] [0, 1, 1, 0] [0, 0, 1, 1] 4 2
()((())) [3, 0, 0, 0] [4, 1, 2, 3] [0, 2, 1, 0] [0, 0, 1, 2] 3 3
(())()() [2, 2, 1, 0] [3, 4, 2, 1] [1, 0, 0, 0] [0, 1, 0, 0] 5 1
(())(()) [2, 2, 0, 0] [3, 4, 1, 2] [1, 0, 1, 0] [0, 1, 0, 1] 4 2
(()())() [1, 2, 1, 0] [2, 4, 3, 1] [1, 1, 0, 0] [0, 1, 1, 0] 4 2
(()()()) [0, 2, 1, 0] [1, 4, 3, 2] [1, 1, 1, 0] [0, 1, 1, 1] 3 3
(()(())) [0, 2, 0, 0] [1, 4, 2, 3] [1, 2, 1, 0] [0, 1, 1, 2] 2 4
((()))() [1, 1, 1, 0] [2, 3, 4, 1] [2, 1, 0, 0] [0, 1, 2, 0] 3 3
((())()) [0, 1, 1, 0] [1, 3, 4, 2] [2, 1, 1, 0] [0, 1, 2, 1] 2 4
((()())) [0, 0, 1, 0] [1, 2, 4, 3] [2, 2, 1, 0] [0, 1, 2, 2] 1 5
(((()))) [0, 0, 0, 0] [1, 2, 3, 4] [3, 2, 1, 0] [0, 1, 2, 3] 0 6
14 14


In [15]:
#For 231 avoiding, clearly strip maps to lehmer code, but exactly how? we need to decide the strip_stat for n digits
#to get a better (i mean perfect) map.
dw = DyckWords(4)
lehmer_set = set()
for d in dw:
    # lehmer123 = dw_alpha_last(d)
    # lehmer132 = dw_beta_last(d)
    # lehmer213 = dw_gamma_last(d)
    lehmer = dw_delta_last(d)
    # lehmer312 = dw_epsilon_last(d)
    # lehmer321 = dw_zeta_last(d)
    lehmer_set.add(tuple(lehmer))
    perm = permutation.from_lehmer_code(lehmer)
    colstat = column_stat(d)
    rowstat = row_stat(d)
    stripright = strip_stat_from_topright(d)
    stripleft = strip_stat_from_bottomleft(d)
    print(d, lehmer, perm, colstat, rowstat, sum(lehmer), len(dyck_to_area_cells(d)))
print(len(dw), len(lehmer_set))

()()()() [0, 0, 0, 0] [1, 2, 3, 4] [0, 0, 0, 0] [0, 0, 0, 0] 0 0
()()(()) [0, 0, 1, 0] [1, 2, 4, 3] [0, 0, 1, 0] [0, 0, 0, 1] 1 1
()(())() [0, 1, 0, 0] [1, 3, 2, 4] [0, 1, 0, 0] [0, 0, 1, 0] 1 1
()(()()) [0, 2, 0, 0] [1, 4, 2, 3] [0, 1, 1, 0] [0, 0, 1, 1] 2 2
()((())) [0, 2, 1, 0] [1, 4, 3, 2] [0, 2, 1, 0] [0, 0, 1, 2] 3 3
(())()() [1, 0, 0, 0] [2, 1, 3, 4] [1, 0, 0, 0] [0, 1, 0, 0] 1 1
(())(()) [1, 0, 1, 0] [2, 1, 4, 3] [1, 0, 1, 0] [0, 1, 0, 1] 2 2
(()())() [2, 0, 0, 0] [3, 1, 2, 4] [1, 1, 0, 0] [0, 1, 1, 0] 2 2
(()()()) [3, 0, 0, 0] [4, 1, 2, 3] [1, 1, 1, 0] [0, 1, 1, 1] 3 3
(()(())) [3, 0, 1, 0] [4, 1, 3, 2] [1, 2, 1, 0] [0, 1, 1, 2] 4 4
((()))() [2, 1, 0, 0] [3, 2, 1, 4] [2, 1, 0, 0] [0, 1, 2, 0] 3 3
((())()) [3, 1, 0, 0] [4, 2, 1, 3] [2, 1, 1, 0] [0, 1, 2, 1] 4 4
((()())) [3, 2, 0, 0] [4, 3, 1, 2] [2, 2, 1, 0] [0, 1, 2, 2] 5 5
(((()))) [3, 2, 1, 0] [4, 3, 2, 1] [3, 2, 1, 0] [0, 1, 2, 3] 6 6
14 14


In [16]:
#For 312, one noticable is lehmer = column stat
dw = DyckWords(4)
lehmer_set = set()
for d in dw:
    # lehmer123 = dw_alpha_last(d)
    # lehmer132 = dw_beta_last(d)
    # lehmer213 = dw_gamma_last(d)
    # lehmer231 = dw_delta_last(d)
    lehmer = dw_epsilon_last(d)
    # lehmer321 = dw_zeta_last(d)
    lehmer_set.add(tuple(lehmer))
    perm = permutation.from_lehmer_code(lehmer)
    colstat = column_stat(d)
    rowstat = row_stat(d)
    stripright = strip_stat_from_topright(d)
    stripleft = strip_stat_from_bottomleft(d)
    print(d, lehmer, perm, colstat, rowstat, sum(lehmer), len(dyck_to_area_cells(d)))
print(len(dw), len(lehmer_set))

()()()() [0, 0, 0, 0] [1, 2, 3, 4] [0, 0, 0, 0] [0, 0, 0, 0] 0 0
()()(()) [0, 0, 1, 0] [1, 2, 4, 3] [0, 0, 1, 0] [0, 0, 0, 1] 1 1
()(())() [0, 1, 0, 0] [1, 3, 2, 4] [0, 1, 0, 0] [0, 0, 1, 0] 1 1
()(()()) [0, 1, 1, 0] [1, 3, 4, 2] [0, 1, 1, 0] [0, 0, 1, 1] 2 2
()((())) [0, 2, 1, 0] [1, 4, 3, 2] [0, 2, 1, 0] [0, 0, 1, 2] 3 3
(())()() [1, 0, 0, 0] [2, 1, 3, 4] [1, 0, 0, 0] [0, 1, 0, 0] 1 1
(())(()) [1, 0, 1, 0] [2, 1, 4, 3] [1, 0, 1, 0] [0, 1, 0, 1] 2 2
(()())() [1, 1, 0, 0] [2, 3, 1, 4] [1, 1, 0, 0] [0, 1, 1, 0] 2 2
(()()()) [1, 1, 1, 0] [2, 3, 4, 1] [1, 1, 1, 0] [0, 1, 1, 1] 3 3
(()(())) [1, 2, 1, 0] [2, 4, 3, 1] [1, 2, 1, 0] [0, 1, 1, 2] 4 4
((()))() [2, 1, 0, 0] [3, 2, 1, 4] [2, 1, 0, 0] [0, 1, 2, 0] 3 3
((())()) [2, 1, 1, 0] [3, 2, 4, 1] [2, 1, 1, 0] [0, 1, 2, 1] 4 4
((()())) [2, 2, 1, 0] [3, 4, 2, 1] [2, 2, 1, 0] [0, 1, 2, 2] 5 5
(((()))) [3, 2, 1, 0] [4, 3, 2, 1] [3, 2, 1, 0] [0, 1, 2, 3] 6 6
14 14


In [17]:
dw = DyckWords(4)
lehmer_set = set()
for d in dw:
    # lehmer123 = dw_alpha_last(d)
    # lehmer132 = dw_beta_last(d)
    # lehmer213 = dw_gamma_last(d)
    # lehmer231 = dw_delta_last(d)
    # lehmer312 = dw_epsilon_last(d)
    lehmer = dw_zeta_last(d)
    lehmer_set.add(tuple(lehmer))
    perm = permutation.from_lehmer_code(lehmer)
    colstat = column_stat(d)
    rowstat = row_stat(d)
    stripright = strip_stat_from_topright(d)
    stripleft = strip_stat_from_bottomleft(d)
    print(d, lehmer, perm, colstat, rowstat, sum(lehmer), len(dyck_to_area_cells(d)))
print(len(dw), len(lehmer_set))

()()()() [0, 0, 0, 0] [1, 2, 3, 4] [0, 0, 0, 0] [0, 0, 0, 0] 0 0
()()(()) [0, 0, 1, 0] [1, 2, 4, 3] [0, 0, 1, 0] [0, 0, 0, 1] 1 1
()(())() [0, 2, 0, 0] [1, 4, 2, 3] [0, 1, 0, 0] [0, 0, 1, 0] 2 1
()(()()) [0, 1, 0, 0] [1, 3, 2, 4] [0, 1, 1, 0] [0, 0, 1, 1] 1 2
()((())) [0, 1, 1, 0] [1, 3, 4, 2] [0, 2, 1, 0] [0, 0, 1, 2] 2 3
(())()() [3, 0, 0, 0] [4, 1, 2, 3] [1, 0, 0, 0] [0, 1, 0, 0] 3 1
(())(()) [2, 0, 1, 0] [3, 1, 4, 2] [1, 0, 1, 0] [0, 1, 0, 1] 3 2
(()())() [2, 0, 0, 0] [3, 1, 2, 4] [1, 1, 0, 0] [0, 1, 1, 0] 2 2
(()()()) [1, 0, 0, 0] [2, 1, 3, 4] [1, 1, 1, 0] [0, 1, 1, 1] 1 3
(()(())) [1, 0, 1, 0] [2, 1, 4, 3] [1, 2, 1, 0] [0, 1, 1, 2] 2 4
((()))() [2, 2, 0, 0] [3, 4, 1, 2] [2, 1, 0, 0] [0, 1, 2, 0] 4 3
((())()) [1, 2, 0, 0] [2, 4, 1, 3] [2, 1, 1, 0] [0, 1, 2, 1] 3 4
((()())) [1, 1, 0, 0] [2, 3, 1, 4] [2, 2, 1, 0] [0, 1, 2, 2] 2 5
(((()))) [1, 1, 1, 0] [2, 3, 4, 1] [3, 2, 1, 0] [0, 1, 2, 3] 3 6
14 14
